## Orchestrator-Workers Workflow
In this workflow, a central LLM dynamically breaks down tasks, delegates them to worker LLMs, and synthesizes their results.

### When to use this workflow
This workflow is well-suited for complex tasks where you can't predict the subtasks needed. The key difference from simple parallelization is its flexibility—subtasks aren't pre-defined, but determined by the orchestrator based on the specific input.

In [1]:
%load_ext autoreload
%autoreload 2

In [90]:
from typing import Dict, List, Optional
from util import llm_call, extract_xml

def parse_tasks(tasks_xml: str) -> List[Dict]:
    """Parse XML tasks into a list of task dictionaries."""
    tasks = []
    current_task = {}
    
    for line in tasks_xml.split('\n'):
        line = line.strip()
        if not line:
            continue
            
        if line.startswith("<task>"):
            current_task = {}
        elif line.startswith("<name>"):
            current_task["name"] = line[6:-7].strip()
        elif line.startswith("<goal>"):
            current_task["goal"] = line[6:-7].strip()
        elif line.startswith("<description>"):
            current_task["description"] = line[12:-13].strip()
        elif line.startswith("</task>"):
            if "description" in current_task:
                if "type" not in current_task:
                    current_task["type"] = "default"
                tasks.append(current_task)
    
    return tasks


class FlexibleOrchestrator:
    """Break down tasks and run them in parallel using worker LLMs."""
    
    def __init__(
        self,
        orchestrator_prompt: str,
        synthetizer_prompt: str,
        worker_prompt: str,
        case_facts: str,
        case_law: str
    ):
        """Initialize with prompt templates."""
        self.orchestrator_prompt = orchestrator_prompt
        self.worker_prompt = worker_prompt
        self.synthetizer_prompt = synthetizer_prompt
        self.case_facts = case_facts
        self.case_law = case_law

    def _format_prompt(self, template: str, **kwargs) -> str:
        """Format a prompt template with variables."""
        try:
            return template.format(**kwargs)
        except KeyError as e:
            raise ValueError(f"Missing required prompt variable: {e}")

    def process(self, task: str, context: Optional[Dict] = None) -> Dict:
        """Process task by breaking it down and running subtasks in parallel."""
        context = context or {}
        
        # Step 1: Get orchestrator response
        orchestrator_input = self._format_prompt(
            self.orchestrator_prompt,
            task=task,
            **context
        )
        orchestrator_response = llm_call(orchestrator_input)
        print("OUTPUT:", orchestrator_response)
        
        # Parse orchestrator response
        analysis = extract_xml(orchestrator_response, "analysis")
        tasks_xml = extract_xml(orchestrator_response, "tasks")
        tasks = parse_tasks(tasks_xml)
        
        print("\n=== ORCHESTRATOR OUTPUT ===")
        print(f"\nANALYSIS:\n{analysis}")
        print(f"\nTASKS:\n{tasks}")
        
        # Step 2: Process each task
        worker_results = []
        for task_info in tasks:
            print('task keys: ', task_info.keys())
            worker_input = self._format_prompt(
                self.worker_prompt,
                #original_task=task,
                task_name=task_info['name'],
                task_goal=task_info['goal'],
                task_description=task_info['description'],
                case_facts = self.case_facts,
                case_law = self.case_law
                #**context
            )
            
            worker_response = llm_call(worker_input)
            result = extract_xml(worker_response, "response")
            
            worker_results.append({
                "name": task_info["name"],
                "goal": task_info["goal"],
                "description": task_info["description"],
                "result": result
            })
            
            print(f"\n=== WORKER RESULT ({task_info['name']}) ===\n{result}\n")
        
        tasks_data = {
            "analysis": analysis,
            "worker_results": worker_results,
        }
    
        sythnsizer_input = self._format_prompt(
            self.synthetizer_prompt,
            original_task=task,
            subtasks = worker_results,
            **context
        )
        print("\n=== ORCHESTRATOR PROMPT ===")
        print(sythnsizer_input)
        sythnsizer_response = llm_call(sythnsizer_input)
        print("OUTPUT:", sythnsizer_response)

        return {
            'orchestrator_response': orchestrator_response,
            'tasks_data': tasks_data,
            'synthetizer_response': sythnsizer_response
        }




### Example Use Case: Marketing Variation Generation



In [91]:
ORCHESTRATOR_PROMPT = """
You are an experienced legal expert specializing in European Court of Human Rights (ECHR) jurisprudence
Analyze this task and break it down into sub tasks that can be completed by different legal experts.

Task: {task}

Return your response in this format:

<analysis>
Explain your understanding of the task and which variations would be valuable.
Focus on how each approach serves different aspects of the task.
</analysis>

List of tasks to achieve the goal, each task should have the following structure:
<tasks>
    <task>
    <name>task name</name>
    <goal>task goal</goal>
    <description>Write a precise, technical version that emphasizes specifications</description>
    </task>
</tasks>
"""

SYNTHESIZER_PROMPT = """
You are an experienced legal expert specializing in European Court of Human Rights (ECHR) jurisprudence
You have been asked to decompose the following task into sub-tasks that can be completed by different legal experts.
Now that the task has been broken down, your task is to use the information provided to provide the final response.

Task: {original_task}

subtasks: {subtasks}

Return your response in this format:
CLASSIFICATION: [NOT KEY CASE or KEY CASE]
REASONING: [Your reasoning here]
"""


WORKER_PROMPT = """
You are an experienced legal expert specializing in European Court of Human Rights (ECHR) jurisprudence
Your task is to provide a detailed response to the following task details:

Task Name: {task_name}
Goal: {task_goal}
Guidelines: {task_description}

Inputs:
case facts: {case_facts}
case reasoning: {case_law}

Return your response in this format:

<response>
Your content here, maintaining the specified style and fully addressing requirements.
</response>
"""

In [92]:
import pandas as pd
from util import load_json
data_path = '/Users/ahmed/Desktop/msc-24/ECHR/echr-processed/'
df1_path = '/Users/ahmed/Desktop/msc-24/TND/kc_classification_data/pre_cutoff_data/df_1.csv'
df4_path = '/Users/ahmed/Desktop/msc-24/TND/kc_classification_data/pre_cutoff_data/df_4.csv'
df1 = pd.read_csv(df1_path)['file_path'].to_list()
df4 = pd.read_csv(df4_path)['file_path'].to_list()

case_path = data_path + df4[0]
case = load_json(case_path)
case_facts = case['facts']  
case_law = case['law']

In [93]:


orchestrator = FlexibleOrchestrator(
    orchestrator_prompt=ORCHESTRATOR_PROMPT,
    synthetizer_prompt=SYNTHESIZER_PROMPT,
    worker_prompt=WORKER_PROMPT,
    case_facts=case_facts,
    case_law=case_law
)

results = orchestrator.process(
    task=F"""
    Classify the importance of the ECHR legal case given the case facts.
    The classification should be one of the two classes:
    
    1. KEY CASE:
    - Makes a significant contribution to the development, clarification, or modification of case law.
    - Establishes new legal principles or substantially modifies existing ones.
    - Has broad implications beyond the immediate case.

    2. NOT KEY CASE:
    - Applies existing case law without significant contributions to legal development.
    - Demonstrates limited implications beyond the immediate dispute.
    
      <Facts>
      {case_facts}.
      </Facts>

      <Law>
      {case_law}.
      </Law>
        

""",
    context={
        "KEY CASE": 
"""
- Contributes significantly to the development, clarification, or modification of ECHR case law.
- Establishes new legal principles or substantially alters existing ones.
- Addresses unique or emerging societal, legal, or procedural trends that could influence future jurisprudence.
- Has broad implications beyond the immediate case.
""" ,
        "target_audience": "Legal Experts and Lawyers in ECHR Jurisdictions",
        "key_features": ["Key case Identification", "Case Law development"]

}
)

OUTPUT: <analysis>
The task involves classifying a legal case from the European Court of Human Rights (ECHR) based on its significance in the evolution of case law. The classification is binary: either the case is deemed a "KEY CASE" for its substantial contributions to legal principles, or it is categorized as "NOT KEY CASE" if it primarily reiterates existing jurisprudence. Variations in approach can include focusing on different procedural aspects, case facts, or broader implications regarding international law.

Understanding the context requires legal experts to analyze case law, identify precedents, and evaluate the impact of the current case on future cases and rulings. They can also assess the procedural integrity of the case and any deviation from existing norms, enhancing the classification's accuracy.

Each expert's focus can highlight distinct elements of the case, allowing for a comprehensive assessment to inform the ultimate classification decision.
</analysis>

<tasks>
 

In [85]:
results

{'orchestrator_response': '<analysis>\nThe task requires evaluating a specific case from the European Court of Human Rights (ECHR) to determine its significance in terms of legal development and implications. The categorization hinges on whether the case serves as a "KEY CASE" that contributes substantially to case law or a "NOT KEY CASE" that merely applies existing jurisprudence with limited broader implications. \n\nVariations that might be valuable include focusing on the nature of the complaints raised (unlawful detention and other Convention violations), the court’s methodologies in assessing jurisdiction concerning Russia’s status, and how closely the case aligns with established case law. Each approach would reveal different insights regarding the evolution of legal principles, trends in human rights jurisprudence in Europe, and the impacts on future cases concerning unlawful detentions and other relevant complaints under the Convention.\n\nThis multi-faceted approach ensures a